# 12. Coleta Multisource de Notícias (2018-2025)

**Objetivo**: Coletar histórico completo de notícias financeiras brasileiras usando múltiplas fontes:

- **GDELT** (principal - cobertura histórica 2018-2025)
- **GNews** (complementar - últimos meses)
- **RSS Feeds** (tempo real - jornais brasileiros)
- **NewsAPI** (opcional - volume adicional)

**Meta**: 30.000+ notícias com deduplicação em NB 13

In [ ]:
# 1. Setup de caminhos e configuração
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path
import json
import time
import warnings
warnings.filterwarnings('ignore')

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])

NB_NAME = "12_data_collection_multisource"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(RAW_PATH, exist_ok=True)

print(f"📂 RAW_PATH: {RAW_PATH}")
print(f"📂 PROC_PATH: {PROC_PATH}")
print(f"⏰ Timestamp: {STAMP}")

Mounted at /content/drive
Base: /content/drive/MyDrive/TCC_USP


In [ ]:
# 2. Importar bibliotecas necessárias
import pandas as pd
import numpy as np
import requests
from typing import List, Dict, Optional

# Verificar instalação de dependências
try:
    import feedparser
    from gnews import GNews
    from bs4 import BeautifulSoup
    print("✅ Todas as dependências instaladas")
except ImportError as e:
    print(f"❌ Dependência faltando: {e}")
    print("Execute: pip install feedparser gnews beautifulsoup4 lxml")
    raise

In [ ]:
# 3. Configuração de período e termos de busca
PERIODO = cfg.get_periodo_estudo()
START_DATE = pd.to_datetime(PERIODO["start"])
END_DATE = pd.to_datetime(PERIODO["end"])

# Termos de busca otimizados para mercado financeiro brasileiro
SEARCH_TERMS = [
    "Ibovespa",
    "Bovespa",
    "B3",
    "bolsa valores Brasil",
    "mercado ações Brasil",
    "economia brasileira",
    "dólar real",
    "Selic taxa juros"
]

print(f"📅 Período: {START_DATE.date()} → {END_DATE.date()}")
print(f"🔍 Termos: {len(SEARCH_TERMS)} queries configuradas")
print(f"📊 Duração: {(END_DATE - START_DATE).days} dias")

🔎 Coletando notícias relacionadas a: Ibovespa OR economia OR bolsa OR ações OR mercado financeiro


In [ ]:
# 4. Importar funções de coleta
from src.utils.multisource_collectors import (
    collect_gdelt_batch,
    collect_gnews,
    collect_rss_feeds,
    collect_newsapi
)

print("✅ Funções de coleta multisource carregadas")

In [ ]:
# 8. Executar coleta NewsAPI (com rate limiting)
df_newsapi = collect_newsapi(SEARCH_TERMS, days_back=30, stamp=STAMP)

if not df_newsapi.empty:
    newsapi_csv = os.path.join(RAW_PATH, "news_newsapi.csv")
    df_newsapi.to_csv(newsapi_csv, index=False, encoding='utf-8-sig')
    print(f"💾 Salvo: {newsapi_csv}")

In [ ]:
# 10. Validação de qualidade dos dados coletados
print("\n" + "="*60)
print("🔍 VALIDAÇÃO DE QUALIDADE")
print("="*60)

# 1. Campos vazios
print("\n📋 Completude dos campos:")
for col in ['title', 'url', 'source', 'published_at']:
    if col in df_all.columns:
        missing = df_all[col].isna().sum()
        pct = (missing / len(df_all)) * 100
        print(f"  {col:20s}: {len(df_all) - missing:6,} preenchidos ({100-pct:.1f}%)")

# 2. Duplicatas aparentes (pré-dedup)
print("\n🔄 Duplicatas aparentes (pré-dedup):")
print(f"  Por URL:    {df_all['url'].duplicated().sum():,}")
print(f"  Por título: {df_all['title'].duplicated().sum():,}")

# 3. Distribuição temporal
if 'published_at_parsed' in df_all.columns:
    df_all['year'] = df_all['published_at_parsed'].dt.year
    print("\n📅 Distribuição por ano:")
    year_counts = df_all['year'].value_counts().sort_index()
    for year, count in year_counts.items():
        if not pd.isna(year):
            print(f"  {int(year)}: {count:,} artigos")

# 4. Validação de período esperado
if len(valid_dates) > 0:
    coverage_days = (valid_dates.max() - valid_dates.min()).days
    expected_days = (END_DATE - START_DATE).days
    coverage_pct = (coverage_days / expected_days) * 100
    
    print(f"\n✅ Cobertura temporal: {coverage_days}/{expected_days} dias ({coverage_pct:.1f}%)")
    
    if coverage_days < expected_days * 0.5:
        print("⚠️ AVISO: Cobertura < 50% do período esperado")
        print("   GDELT pode ter limitações em algumas regiões/idiomas")
    if len(df_all) < 5000:
        print(f"⚠️ AVISO: Volume total ({len(df_all):,}) abaixo do ideal (30.000+)")
        print("   Considere ajustar termos de busca ou adicionar mais fontes")
    else:
        print("✅ Validação OK - volume e cobertura adequados")

print("\n" + "="*60)
print(f"✅ COLETA MULTISOURCE CONCLUÍDA")
print("="*60)

In [ ]:
# 9. Consolidar todos os datasets
dfs = []

if not df_gdelt.empty:
    dfs.append(df_gdelt)
if not df_gnews.empty:
    dfs.append(df_gnews)
if not df_rss.empty:
    dfs.append(df_rss)
if not df_newsapi.empty:
    dfs.append(df_newsapi)

if not dfs:
    raise ValueError("❌ Nenhuma fonte retornou dados. Verifique configurações.")

df_all = pd.concat(dfs, ignore_index=True)

print("\n" + "="*60)
print("📊 ESTATÍSTICAS DE COLETA MULTISOURCE")
print("="*60)
print(f"\n🌐 Total de artigos brutos: {len(df_all):,}")
print(f"\n📈 Por fonte:")
print(df_all['source_type'].value_counts())
print(f"\n📅 Período coberto:")
if 'published_at' in df_all.columns:
    df_all['published_at_parsed'] = pd.to_datetime(df_all['published_at'], errors='coerce')
    valid_dates = df_all['published_at_parsed'].dropna()
    if len(valid_dates) > 0:
        print(f"  Primeira: {valid_dates.min()}")
        print(f"  Última:   {valid_dates.max()}")
        print(f"  Dias:     {(valid_dates.max() - valid_dates.min()).days}")

print(f"\n🔍 Top 10 fontes:")
print(df_all['source'].value_counts().head(10))

# Salvar dataset consolidado (PRÉ-DEDUP)
raw_consolidated = os.path.join(RAW_PATH, f"news_multisource_raw_{STAMP}.parquet")
df_all.to_parquet(raw_consolidated, index=False, engine='pyarrow')
print(f"\n💾 Dataset consolidado (PRÉ-DEDUP): {raw_consolidated}")
print(f"📦 Shape: {df_all.shape}")
print(f"\n⏭️ Próximo passo: Executar Notebook 13 (ETL + Deduplicação)")

## 📊 Consolidação e Estatísticas Finais

## 🔑 FONTE D: NewsAPI (Opcional - Volume Adicional)

In [ ]:
# 7. Executar coleta RSS Feeds
df_rss = collect_rss_feeds(STAMP)

# Salvar por fonte
for source in df_rss['source'].unique():
    df_source = df_rss[df_rss['source'] == source]
    rss_csv = os.path.join(RAW_PATH, f"news_rss_{source}.csv")
    df_source.to_csv(rss_csv, index=False, encoding='utf-8-sig')
    print(f"💾 Salvo: {rss_csv}")

## 📡 FONTE C: RSS Feeds (Tempo Real - Jornais Brasileiros)

In [ ]:
# 6. Executar coleta GNews
df_gnews = collect_gnews(SEARCH_TERMS, max_results=100, stamp=STAMP)

# Salvar
gnews_csv = os.path.join(RAW_PATH, "news_gnews.csv")
df_gnews.to_csv(gnews_csv, index=False, encoding='utf-8-sig')
print(f"💾 Salvo: {gnews_csv}")

## 📰 FONTE B: GNews (Complementar - Últimos Meses)

In [ ]:
# 5. Executar coleta GDELT (Principal - 2018-2025)
df_gdelt = collect_gdelt_batch(START_DATE, END_DATE, SEARCH_TERMS, STAMP)

# Salvar checkpoint GDELT
gdelt_csv = os.path.join(RAW_PATH, "news_gdelt_2018_2025.csv")
gdelt_json = os.path.join(RAW_PATH, "news_gdelt_2018_2025.json")

df_gdelt.to_csv(gdelt_csv, index=False, encoding='utf-8-sig')
df_gdelt.to_json(gdelt_json, orient='records', force_ascii=False, indent=2)

print(f"💾 Salvo: {gdelt_csv}")
print(f"💾 Salvo: {gdelt_json}")
print(f"📊 Shape: {df_gdelt.shape}")

## 🌐 FONTE A: GDELT Project (Principal - 2018-2025)